# Production-Grade Prompting, Agents & Tool Use

**Module 2 lab** — the sequel to *MSO Foundations*. Writing code that uses Claude is different from using Claude to write code. In this lab we take a contract-review prototype and harden it against the failures that only show up in production.

Each chapter targets **one failure mode** and shows the fix:

| # | Chapter | Production failure it prevents |
|---|---|---|
| 1 | Setup | — |
| 2 | Prompting craft | Output in the wrong shape breaks the parser |
| 3 | Extended thinking | Paying for reasoning you don't need |
| 4 | Tool use & schema design | Claude picks the wrong tool; the loop breaks |
| 5 | Streaming | A dropped stream corrupts history |
| 6 | Context engineering | The context window fills at turn 8 |
| 7 | Agent construction | An agent makes an irreversible change unchecked |
| 8 | Agent memory | State is lost (or bloats every call) between sessions |
| 9 | Multimodal & batch | Image costs and rate limits blow up at scale |
| 10 | Debug challenge | All of the above in one buggy agent |

### Prerequisite
Download [AWS1.pdf](https://drive.google.com/file/d/1XSe2pSsGN1ssAbif92rvb80AnHb_Ni0F/view?usp=sharing) (the short AWS contract from the MSO lab). You'll upload it in Chapter 1.

## Chapter 1 — Setup

In [ ]:
!pip install -q -U anthropic PyMuPDF pyboxen pillow

In [ ]:
import os, json, math, base64, asyncio, time
from getpass import getpass

import anthropic
import fitz  # PyMuPDF
from pyboxen import boxen

os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key: ")
client = anthropic.Anthropic()

HAIKU, SONNET = "claude-haiku-4-5", "claude-sonnet-4-6"


def show(title, text, color="cyan"):
    print(boxen(str(text), title=title, color=color, padding=1))


def text_of(response):
    """Join all text blocks of a response."""
    return "".join(b.text for b in response.content if b.type == "text")

Upload `AWS1.pdf`, then load it page by page — we'll reuse it in every chapter.

In [ ]:
from google.colab import files
files.upload()

with fitz.open("/content/AWS1.pdf") as doc:
    pages = [p.get_text() for p in doc]
contract_text = "\n".join(pages)
print(f"{len(pages)} pages · {len(contract_text):,} characters")

## Chapter 2 — Prompting Craft

A prompt that works once in a chat often breaks in production on inputs you didn't test. **Don't add more words — diagnose which structural piece is missing:**

| What you observe | What's missing | Fix |
|---|---|---|
| Wrong *shape* (a sentence instead of a label) | **Output constraint** | State exact form, allowed values, "nothing else" |
| Scope/tone drifts across turns | **System prompt** | Persistent role + rules |
| Right task, invented structure | **Few-shot examples** | Show one correct input → output pair |
| Instructions blend with data | **XML tags** | `<clause>…</clause>` boundaries |

**The task:** a router needs one of four labels for each contract clause. We'll measure how often the output is *usable* — one valid label, nothing else.

In [ ]:
LABELS = {"TERMINATION", "PAYMENT", "LIABILITY", "OTHER"}

clauses = [
    "Either party may terminate this Agreement for convenience on 30 days' written notice.",
    "Customer shall pay all fees within 30 days of the invoice date.",
    "Neither party's aggregate liability shall exceed the fees paid in the prior 12 months.",
    "This Agreement is governed by the laws of the State of Washington.",
    "We may suspend your access if you breach this Agreement, and fees continue to accrue during suspension.",
]


def valid_rate(system, wrap, runs=3):
    """Classify every clause `runs` times; return (share of valid outputs, sample outputs)."""
    outs = []
    for clause in clauses:
        for _ in range(runs):
            r = client.messages.create(
                model=HAIKU, max_tokens=60, system=system,
                messages=[{"role": "user", "content": wrap(clause)}],
            )
            outs.append(text_of(r).strip())
    ok = sum(o in LABELS for o in outs)
    return ok / len(outs), outs


bare_rate, bare_outs = valid_rate("You are a clause classifier. Classify the clause.", lambda c: c)
show("❌ Bare prompt", f"Valid outputs: {bare_rate:.0%}\n\nSamples:\n" + "\n".join("• " + o[:80] for o in bare_outs[:4]), "red")

Now apply the fix: **system prompt + output constraint + XML-wrapped few-shot examples** (including the ambiguous case).

In [ ]:
SYSTEM = """You are a contract clause classifier.
Classify each clause into exactly one of: TERMINATION, PAYMENT, LIABILITY, OTHER.
Return only the label. No other text.

<sample_input>Either party may end this agreement with 60 days' notice.</sample_input>
<ideal_output>TERMINATION</ideal_output>
<sample_input>Invoices are due net 45.</sample_input>
<ideal_output>PAYMENT</ideal_output>
<sample_input>We may suspend service for breach, and fees still accrue.</sample_input>
<ideal_output>TERMINATION</ideal_output>"""

good_rate, good_outs = valid_rate(SYSTEM, lambda c: f"<clause>{c}</clause>")
show("✅ Constrained prompt", f"Valid outputs: {good_rate:.0%}\n\nSamples: {sorted(set(good_outs))}", "green")

### Move the contract into the API: structured outputs
A prompt is a *request*; a **JSON schema** is *enforced* (constrained decoding — invalid tokens can't be generated).

- `output_config.format` → constrains the **final response**
- `strict: true` on a tool → constrains the **tool arguments** (used in Chapter 4)

Costs: first call on a new schema is slower (grammar compile, cached 24 h), a few extra input tokens, and you must still check `stop_reason` (`refusal` and `max_tokens` can still produce non-matching output). Not compatible with message prefilling.

In [ ]:
schema = {
    "type": "object",
    "properties": {
        "category": {"type": "string", "enum": sorted(LABELS)},
        "summary": {"type": "string"},
    },
    "required": ["category", "summary"],
    "additionalProperties": False,
}

r = client.messages.create(
    model=SONNET, max_tokens=300,
    output_config={"format": {"type": "json_schema", "schema": schema}},
    messages=[{"role": "user", "content": f"Classify and summarize in one sentence: <clause>{clauses[4]}</clause>"}],
)

assert r.stop_reason == "end_turn", f"Don't parse: stop_reason={r.stop_reason}"  # always check
show("🧱 Schema-enforced output", json.loads(text_of(r)), "green")

> **Rule of thumb:** name the failure → add the one matching technique → re-run. If three re-prompts in a row didn't help, stop adding text. Also **simplify**: a "summarize this paragraph" prompt needs none of these.

## Chapter 3 — Extended Thinking

Prompting shapes *what* Claude produces; **thinking** shapes *how much work* it does first. On current models thinking is **adaptive** — Claude decides how much to think and you steer depth with `effort` (the old `budget_tokens` is deprecated and errors on the newest models).

| Task | Thinking? |
|---|---|
| Multi-step reasoning, planning dependent actions | ✅ On, effort matched to difficulty |
| Classification, extraction, format conversion | ❌ Off — a constrained prompt is cheaper and as accurate |
| Agent loops with tools | ✅ On — but obey the **carry-back rule** below |

Thinking tokens bill like output tokens. Compare cost on the same hard question:

In [ ]:
hard_q = (
    "Using the contract below, list every way the contract can end or access can be suspended, "
    "who can trigger each, and which one gives the customer the least protection. Cite sections.\n\n" + contract_text
)

runs = {
    "thinking OFF": dict(thinking={"type": "disabled"}),
    "adaptive · effort=high": dict(
        thinking={"type": "adaptive", "display": "summarized"}, output_config={"effort": "high"}
    ),
}

for name, kwargs in runs.items():
    r = client.messages.create(model=SONNET, max_tokens=6000,
                               messages=[{"role": "user", "content": hard_q}], **kwargs)
    show(f"{name} · blocks={[b.type for b in r.content]} · output_tokens={r.usage.output_tokens}",
         text_of(r)[:700] + " …", "yellow" if "OFF" in name else "purple")

### The carry-back rule
When thinking is on **and** you use tools, every `thinking` block must go back to the API **exactly as received** on the next turn (it carries a cryptographic signature). Editing, summarizing or dropping it → the request is rejected. Redacted thinking blocks too.

> Worried about context bloat from reasoning? Don't strip blocks — manage context (Chapter 6).

In our loops we'll simply append `response.content` untouched, which satisfies this automatically.

## Chapter 4 — Tool Use & Schema Design

**Claude never runs your tools.** It reads your schemas, decides which fits, and returns a `tool_use` block; *your code* runs it and sends back a `tool_result`.

```text
define schema → send message → tool_use block → YOU execute → tool_result → Claude continues
```

| Block | Rule |
|---|---|
| `text` | Keep it when you append the assistant turn |
| `tool_use` | Has a unique `id` |
| `tool_result` | Must be in the **very next user turn**, same `tool_use_id` (`is_error: true` on failure) |
| `thinking` | Pass back unchanged |

A schema has `name`, `description` and `input_schema`. **The description decides tool selection** — write when to use it *and when not to*.

Let's give Claude two real tools over our contract:

In [ ]:
def search_contract(keyword):
    """Return up to 5 short snippets (with page numbers) that contain the keyword."""
    hits = []
    for n, text in enumerate(pages, 1):
        i = text.lower().find(keyword.lower())
        if i >= 0:
            hits.append(f"[page {n}] …{text[max(0, i - 120): i + 200]}…".replace("\n", " "))
    return "\n".join(hits[:5]) or "No matches."


def get_page(page):
    """Return the text of one page (capped at 2,500 characters to protect context)."""
    if not 1 <= page <= len(pages):
        raise ValueError(f"page must be 1–{len(pages)}")
    return pages[page - 1][:2500]


IMPL = {"search_contract": search_contract, "get_page": get_page}

tools_good = [
    {
        "name": "search_contract",
        "description": (
            "Search the contract for a keyword and return short snippets with page numbers. "
            "Use this when you don't yet know where a topic appears. "
            "Do not use this if you already know the page number — use get_page instead."
        ),
        "input_schema": {"type": "object", "properties": {"keyword": {"type": "string"}}, "required": ["keyword"]},
    },
    {
        "name": "get_page",
        "description": (
            "Return the full text of one contract page. Use this only when you already know the page number "
            "(for example from a search result). Do not use this to look for a topic."
        ),
        "input_schema": {"type": "object", "properties": {"page": {"type": "integer"}}, "required": ["page"]},
    },
]

# Same tools with overlapping, vague descriptions — Claude can't tell them apart
tools_bad = [
    {**t, "description": "Use this to find information in the contract."} for t in tools_good
]

**Which tool does Claude pick first?** Same questions, vague vs. specific descriptions. (Vague descriptions don't fail every single time — they fail on boundary cases, and get worse as you add more tools.)

In [ ]:
def first_tool(tools, question):
    r = client.messages.create(model=HAIKU, max_tokens=300, tools=tools,
                               messages=[{"role": "user", "content": question}])
    return next((b.name for b in r.content if b.type == "tool_use"), "none")


tests = [("Where does the contract discuss termination?", "search_contract"),
         ("Show me everything on page 3.", "get_page")]

for label, tools in [("vague", tools_bad), ("specific", tools_good)]:
    picks = [(first_tool(tools, q), expected) for q, expected in tests]
    show(f"{label} descriptions", "\n".join(f"picked {p:16} expected {e}" for p, e in picks),
         "red" if label == "vague" else "green")

### The tool-use loop
Run tools until `stop_reason != "tool_use"`. All `tool_use` blocks from one turn (Claude may call several **in parallel**) are answered together in **one** user message.

In [ ]:
def run_tools(question, tools=tools_good, max_iters=6):
    messages = [{"role": "user", "content": question}]
    for _ in range(max_iters):
        r = client.messages.create(model=SONNET, max_tokens=1500, tools=tools, messages=messages)
        messages.append({"role": "assistant", "content": r.content})   # keep ALL blocks, unchanged
        if r.stop_reason != "tool_use":
            return text_of(r), messages
        results = []
        for b in r.content:
            if b.type != "tool_use":
                continue
            try:
                out, err = IMPL[b.name](**b.input), False
            except Exception as e:                                       # report failures to Claude
                out, err = f"Error: {e}", True
            print(f"🔧 {b.name}({b.input}){'  ❌' if err else ''}")
            results.append({"type": "tool_result", "tool_use_id": b.id, "content": str(out), "is_error": err})
        messages.append({"role": "user", "content": results})        # ALL results, one user turn
    raise RuntimeError("max_iters reached")


answer, history = run_tools("What notice period applies to termination for convenience, and on which page is it?")
show("Answer", answer, "green")

### What a broken pairing looks like
Same history, but the `tool_result` carries the **wrong id**. The API rejects it — no prompt tweak can fix a structural error.

In [ ]:
first_assistant = next(m for m in history if m["role"] == "assistant"
                       and any(b.type == "tool_use" for b in m["content"]))
broken = [
    history[0],
    first_assistant,
    {"role": "user", "content": [{"type": "tool_result", "tool_use_id": "toolu_WRONG", "content": "Page text…"}]},
]

try:
    client.messages.create(model=SONNET, max_tokens=300, tools=tools_good, messages=broken)
except anthropic.BadRequestError as e:
    show("🚫 Validation error", e.message, "red")

**Schema design cheat-sheet**

| Decision | Guidance |
|---|---|
| Required fields | Only if the call makes no sense without them — else Claude invents values |
| Optional fields | Sensible defaults; Claude may omit what it doesn't know |
| Description length | 3–4 sentences: what, when, when *not*, what it returns |
| Dependent calls | Separate turns. Independent calls run in parallel (`disable_parallel_tool_use` forces one) |
| Near-duplicate tools | Add exclusions — or merge into one tool with a `type` parameter |
| `strict: true` | Validates the arguments against your schema before your code runs |

**MCP** (Module 3) lets you *connect* maintained tool servers instead of hand-writing schemas; the loop is identical, but every connected server adds tool definitions to context.

## Chapter 5 — Streaming Without Corrupting State

Streaming sends **events**; *you* assemble the message.

| Event | Your handler |
|---|---|
| `message_start` | Create empty content list |
| `content_block_start` | Open a slot (a `tool_use` block opens with name + id, **no input yet**) |
| `content_block_delta` | Append the fragment (tool input arrives as **partial JSON**) |
| `content_block_stop` | Block complete — **first moment** tool JSON can be parsed |
| `message_delta` | Record `stop_reason` |
| `message_stop` | Whole message done — **only now** commit to history |

> A stream *ending* ≠ a message *completing*. If the connection drops mid-`tool_use`, a half-built block in history breaks the **next** request — and the error points at the wrong place.

In [ ]:
def stream_turn(messages, tools, drop_after=None):
    """Stream one turn. Returns (content, stop_reason), or (None, 'interrupted') if the stream never reached message_stop."""
    blocks, stop_reason, finished, seen = {}, None, False, 0
    with client.messages.create(model=SONNET, max_tokens=1000, tools=tools, messages=messages, stream=True) as stream:
        for ev in stream:
            seen += 1
            if ev.type == "content_block_start":
                b = ev.content_block
                blocks[ev.index] = {"type": b.type, "id": getattr(b, "id", None), "name": getattr(b, "name", None), "text": "", "json": ""}
            elif ev.type == "content_block_delta":
                d, blk = ev.delta, blocks[ev.index]
                if d.type == "text_delta":
                    blk["text"] += d.text
                    print(d.text, end="", flush=True)
                elif d.type == "input_json_delta":
                    blk["json"] += d.partial_json            # NOT valid JSON yet
            elif ev.type == "content_block_stop":
                blk = blocks[ev.index]
                if blk["type"] == "tool_use":
                    blk["input"] = json.loads(blk["json"] or "{}")   # safe only now
            elif ev.type == "message_delta":
                stop_reason = ev.delta.stop_reason
            elif ev.type == "message_stop":
                finished = True
            if drop_after and seen >= drop_after:               # simulate a dropped connection
                break
    if not finished:
        return None, "interrupted"
    content = []
    for i in sorted(blocks):
        b = blocks[i]
        content.append({"type": "text", "text": b["text"]} if b["type"] == "text"
                       else {"type": "tool_use", "id": b["id"], "name": b["name"], "input": b["input"]})
    return content, stop_reason


q = [{"role": "user", "content": "Find the termination for convenience clause, then explain it in 2 sentences."}]

print("— healthy stream —")
content, reason = stream_turn(q, tools_good)
print(f"\n\n→ stop_reason={reason}, blocks={[b['type'] for b in content]}")

In [ ]:
print("— stream dropped mid-way —")
history2 = list(q)
content, reason = stream_turn(history2, tools_good, drop_after=8)

if content is None:                       # gate on message_stop
    print(f"\n\n→ {reason}: partial turn DISCARDED, history untouched ({len(history2)} message). Safe to retry.")
else:
    history2.append({"role": "assistant", "content": content})

**Handler rules:** (1) act on a block only after `content_block_stop`; (2) append to history only after `message_stop`; (3) on interruption discard and retry from the last complete turn; (4) check `stop_reason == "tool_use"` before continuing a loop. For short, backend-only responses a non-streamed call is simpler and removes this whole risk.

## Chapter 6 — Context Engineering

The context window is a **fixed budget** holding system prompt + history + tool definitions + every tool result + the answer. Tool outputs are the fastest spenders: in production they're typically **3–5× longer** than your test fixtures, so a session that survives 50 turns in dev can hit its ceiling at turn 8 — and the symptom looks like *wrong tool selection*, not overflow.

**Model choice:** start with Sonnet; move to Opus only if an eval says Sonnet misses your bar; move to Haiku only if an eval says the drop is acceptable.

Let's measure instead of guessing — count tokens as an agent reads page after page:

In [ ]:
def n_tokens(text):
    return client.messages.count_tokens(model=SONNET, messages=[{"role": "user", "content": text}]).input_tokens


BUDGET = 6_000          # a deliberately small team-imposed budget, to see the effect
used = n_tokens(SYSTEM)
for turn, page in enumerate(range(1, len(pages) + 1), 1):
    used += n_tokens(get_page(page))
    print(f"turn {turn:2}: +page {page:2} → {used:6,} tokens {'█' * (used // 400)}")
    if used > BUDGET:
        print(f"\n🚨 Budget of {BUDGET:,} exceeded at turn {turn}. Time to prune/compact — before the symptoms start.")
        break

### Four ways to stay in budget

| Strategy | What it does | Use when | You lose |
|---|---|---|---|
| **Pruning** | Rewind / drop spent tool results | Dead-end path, finished lookups | Work after the rewind point |
| **Compaction** (`/compact`, server-side, or your own summarizer) | Summarize history | Near the ceiling, same task continues | Anything the summary omits |
| **Clearing** (`/clear`, new session) | Empty context | Next task is unrelated | Everything not persisted |
| **Subagent handoff** | Isolated context, returns a summary | Self-contained / exploratory subtask | Visibility into its steps |

**Compaction quality = summarizer prompt quality.** Compare:

In [ ]:
transcript = """
user: Please update the rate limit in config/prod.yaml from 100 to 500.
assistant: I edited config/prod.yaml (rate_limit: 500) and config/staging.yaml (rate_limit: 500).
user: validate_config failed: 'burst' must be <= rate_limit.
assistant: Fixed by setting burst: 400 in config/prod.yaml. validate_config now passes.
user: Don't touch staging, that was a mistake. Revert it.
assistant: Reverted config/staging.yaml to rate_limit: 100.
"""

generic = "Summarize the conversation so far.\n" + transcript
specific = ("Summarize the conversation, preserving: all file paths modified, all decisions made, "
            "and any errors encountered and how they were resolved.\n" + transcript)

for name, p in [("generic summarizer", generic), ("specific summarizer", specific)]:
    r = client.messages.create(model=HAIKU, max_tokens=300, messages=[{"role": "user", "content": p}])
    show(name, text_of(r), "red" if name.startswith("generic") else "green")

### Prompt caching: pay once for the stable prefix
Mark the last block you want cached with `cache_control: {"type": "ephemeral"}` (up to 4 breakpoints; order is tools → system → messages). The prefix must match **exactly**. Default TTL is 5 min (`"ttl": "1h"` opt-in), and there's a minimum size (~1,024 tokens on most models) — shorter prompts silently don't cache.

In [ ]:
system_blocks = [
    {"type": "text", "text": "You answer questions about the contract below.\n\n" + contract_text,
     "cache_control": {"type": "ephemeral"}}
]

for i, q in enumerate(["Who can terminate the agreement?", "What happens to customer content after termination?"], 1):
    r = client.messages.create(model=SONNET, max_tokens=200, system=system_blocks,
                               messages=[{"role": "user", "content": q}])
    u = r.usage
    print(f"request {i}: cache_write={u.cache_creation_input_tokens or 0:>6,}  cache_read={u.cache_read_input_tokens or 0:>6,}  uncached_input={u.input_tokens:>5,}")

Request 1 **writes** the cache; request 2 **reads** it at a fraction of the cost. For multi-turn sessions with a stable system prompt + tool schemas, this is the highest-leverage cost cut. `count_tokens` (used above) lets you gate oversized requests *before* they fail.

**Subagents** start clean: they get only the task + the context you hand over — and they don't inherit your skills.

## Chapter 7 — Building a Production Agent

An **agent** = a multi-step tool-use loop + managed context + a goal. Agents add coordination cost and failure surface, so decide *first*:

| Choose a **workflow** when… | Choose an **agent** when… |
|---|---|
| You can list the exact steps in code | You can specify goal + tools, but not the path |
| Error cost is real, step guardrails matter | Inputs vary unpredictably |
| Every run follows the same sequence | The task needs creative sequencing of tools |

> Progression: single API call → workflow → agent. Move up only when the simpler pattern can't cope.

**Who runs the loop? (same pattern, three wiring paths)**

| Path | You own | Pick when |
|---|---|---|
| **Raw Messages API loop** | Everything | Full control; learning the loop |
| **Agent SDK** | Tool execution + app (SDK runs loop in your process) | You want Claude Code's loop; set `settingSources` explicitly |
| **Managed Agents** (beta) | App + agent definition (Anthropic runs loop + sandbox) | Long-running work. ⚠️ Server-side state → not eligible for ZDR / HIPAA BAA |

**Loop checklist:** ① tools registered ② system prompt scoped to the task ③ every `tool_use` answered ④ **a human gate before irreversible actions** ⑤ explicit exit conditions.

**Where does a human gate go?** Ask *"what's the worst outcome if this step runs unchecked?"* → before destructive/write/send calls, after a plan, and on unexpected output.

Below, `send_report` is irreversible (it "emails" a result), so it needs approval:

In [ ]:
sent_reports = []


def send_report(to, body):
    sent_reports.append((to, body))          # stand-in for a real, irreversible email
    return f"Report sent to {to}."


IMPL["send_report"] = send_report

agent_tools = tools_good + [{
    "name": "send_report",
    "description": (
        "Email a final report to a recipient. This is irreversible. Use it once, only after the analysis "
        "is complete. Do not use it for drafts or intermediate notes."
    ),
    "input_schema": {
        "type": "object",
        "properties": {"to": {"type": "string"}, "body": {"type": "string"}},
        "required": ["to", "body"],
    },
}]

NEEDS_APPROVAL = {"send_report"}


def human_approves(name, args):
    show("✋ Human checkpoint", f"Agent wants to run {name}\n\n{json.dumps(args, indent=2)[:600]}", "yellow")
    return input("Approve? (y/N): ").strip().lower() == "y"


def run_agent(goal, max_iters=10):
    messages = [{"role": "user", "content": goal}]
    system = ("You are a contract-review agent. Tools: search_contract, get_page, send_report. "
              "Research with the read tools, then send exactly one report. Stop when the report is sent.")
    for i in range(max_iters):                                     # exit condition #1: iteration cap
        r = client.messages.create(model=SONNET, max_tokens=1500, system=system, tools=agent_tools, messages=messages)
        messages.append({"role": "assistant", "content": r.content})
        if r.stop_reason != "tool_use":                            # exit condition #2: model is done
            return text_of(r), messages
        results = []
        for b in r.content:
            if b.type != "tool_use":
                continue
            if b.name in NEEDS_APPROVAL and not human_approves(b.name, b.input):
                results.append({"type": "tool_result", "tool_use_id": b.id, "is_error": True,
                                "content": "Rejected by human reviewer. Do not retry; ask what to change."})
                continue
            try:
                out, err = str(IMPL[b.name](**b.input)), False
            except Exception as e:
                out, err = f"Error: {e}", True
            print(f"🔧 [{i}] {b.name}({str(b.input)[:80]})")
            results.append({"type": "tool_result", "tool_use_id": b.id, "content": out, "is_error": err})
        messages.append({"role": "user", "content": results})
    raise RuntimeError("Agent hit max_iters without finishing")


final, _ = run_agent("Find the termination notice rules and email a 3-bullet summary to legal@example.com.")
show("Agent result", final, "green")
print("Reports actually sent:", len(sent_reports))

Run it twice — answer **y** once and **N** once — and watch how rejection flows back to Claude as an *error result* rather than a crash.

> **Over-tooling** is the common production bug: registering every tool "just in case" degrades routing. Start with the minimum set and add only for a confirmed gap.

**Regulated data picks your endpoint *before* you design anything.** HIPAA → a BAA-covered configuration (not Console/Workbench/consumer plans). EU residency → Bedrock/Vertex with a pinned region. FedRAMP → Claude for Government, Bedrock GovCloud, or Vertex Assured Workloads. Privilege → direct API through the firm's logged gateway. (Module 4 goes deeper on security.)

## Chapter 8 — Agent Memory

Memory scope is decided by **the shape of the sessions**, not by what's easiest to code:

| Scope | Persists | Cost | Use when |
|---|---|---|---|
| **In-context** | Within one session | Tokens grow every turn | Short, one-off sessions |
| **External storage** | Across sessions/users | Retrieval latency + code | Multi-day threads, shared state |
| **Summarized** | Condensed history | Cheap, but lossy | Long conversations |
| **Stateless** | Nothing | None | Self-contained jobs |

The classic failure: dev used one long session; production runs many short ones and the in-context history is already 40k tokens at session 4. Refactoring then takes hours; deciding at design time takes minutes.

Here's **external storage + a targeted summarizer** in miniature:

In [ ]:
MEMORY_FILE = "/content/agent_memory.json"


def save_memory(case_id, summary):
    mem = json.load(open(MEMORY_FILE)) if os.path.exists(MEMORY_FILE) else {}
    mem[case_id] = summary
    json.dump(mem, open(MEMORY_FILE, "w"), indent=2)


def load_memory(case_id):
    return json.load(open(MEMORY_FILE)).get(case_id, "") if os.path.exists(MEMORY_FILE) else ""


# ── Session 1 (illustrative transcript) ──
s1 = [
    {"role": "user", "content": "Review the AWS termination terms for case C-17. Key concern: data deletion."},
    {"role": "assistant", "content": "After termination the customer has 30 days to retrieve content. Open question: what happens to backups?"},
]
summary = text_of(client.messages.create(model=HAIKU, max_tokens=250, messages=[{"role": "user", "content":
    "Summarize for the next session, preserving: the case id, the user's concern, decisions made, open questions.\n\n"
    + json.dumps(s1)}]))
save_memory("C-17", summary)

# ── Session 2 (fresh process, no history) ──
r = client.messages.create(model=HAIKU, max_tokens=200,
    system="Prior context for this case:\n" + load_memory("C-17"),
    messages=[{"role": "user", "content": "Remind me what we concluded and what is still open."}])
show("Session 2 — recalls Session 1 from storage", text_of(r), "green")

### Skills: reusable instructions that load *on demand*
A **Skill** is a `SKILL.md` (frontmatter `name` + `description`, then steps). Claude reads only names/descriptions up front and loads the body when a request matches — so it costs almost nothing until needed.

| Pattern | Loads | Context cost | Best for |
|---|---|---|---|
| **Skill** | On description match | Low | Task-specific procedures |
| **CLAUDE.md** | Every session | Fixed, always | Always-on project rules |
| **In-context** | Every turn | Grows with session | One-off, short work |

Skills on the Messages API are **beta** (headers `code-execution-2025-08-25` + `skills-2025-10-02`; runs in a code-execution container). **Subagents don't inherit skills** — list them explicitly. Verify headers against current docs before building.

## Chapter 9 — Multimodal & Batch Ingestion

**Images cost tokens before Claude reads a single word:** each 28×28-pixel patch ≈ 1 token → `⌈w/28⌉ × ⌈h/28⌉`. Each model has a max resolution; larger images are downscaled first. Compute cost from the *largest* production image, not your test thumbnails — a resize step is a 10-minute fix before launch and a long one after.

In [ ]:
from PIL import Image
import io


def image_block(w, h):
    buf = io.BytesIO()
    Image.new("RGB", (w, h), "white").save(buf, "PNG")
    return {"type": "image", "source": {"type": "base64", "media_type": "image/png",
            "data": base64.standard_b64encode(buf.getvalue()).decode()}}


baseline = n_tokens("Describe this image.")
for w, h in [(280, 280), (1000, 1000), (1400, 1400)]:
    est = math.ceil(w / 28) * math.ceil(h / 28)
    real = client.messages.count_tokens(model=SONNET, messages=[{"role": "user", "content": [
        image_block(w, h), {"type": "text", "text": "Describe this image."}]}]).input_tokens - baseline
    print(f"{w}×{h}: formula ≈ {est:5,} tokens | API reports ≈ {real:5,} (limits/downscaling may differ)")

### How to send an asset

| Method | Overhead | Use when |
|---|---|---|
| **Inline base64** | Full payload every request | One-off image |
| **URL** | None, but must be public, stable, reachable | Already hosted at a stable URL |
| **Files API** (beta; not on Bedrock/Vertex) | Upload once, then just a `file_id` | Same asset reused, or asset management separate from inference |

PDFs use a `document` block (same three source types; optional `title`/`context`). Everything you learned about prompting still applies — and images add ambiguity, so say how to handle it ("if objects overlap, describe each and note the overlap").

### Batches: the right tool for offline volume
Looping over the sync API **is not batching** — it's serial calls that still hit rate limits. The **Message Batches API** takes up to 100,000 requests / 256 MB, runs asynchronously (≤24 h), costs less per token, and returns results in **arbitrary order** → match by `custom_id`.

| Use case | API |
|---|---|
| User waits for a classification | Sync |
| Nightly job on 5,000 records | **Batch** |
| Eval of a new prompt on 2,000 examples | **Batch** |
| Chat reply | Sync |

In [ ]:
batch = client.messages.batches.create(requests=[
    {"custom_id": f"clause-{i}",
     "params": {"model": HAIKU, "max_tokens": 20, "system": SYSTEM,
                "messages": [{"role": "user", "content": f"<clause>{c}</clause>"}]}}
    for i, c in enumerate(clauses)
])
print("Submitted batch", batch.id, "— this can take a few minutes…")

while (status := client.messages.batches.retrieve(batch.id)).processing_status != "ended":
    time.sleep(10)

results = {item.custom_id: text_of(item.result.message).strip()
           for item in client.messages.batches.results(batch.id) if item.result.type == "succeeded"}
for i, c in enumerate(clauses):                       # re-join by custom_id, NOT by position
    print(f"{results.get(f'clause-{i}', '?'):12} ← {c[:60]}…")

## Chapter 10 — Debug Challenge

This agent has **four planted bugs**, one per layer: *schema*, *streaming*, *context*, *memory*. Find them before opening the answer.

```python
tools = [{
    "name": "get_customer_data",
    "description": "Gets data.",
    "input_schema": {"type": "object", "properties": {"id": {"type": "string"}}, "required": ["id"]},
}]

def run_agent(user_request, session_history):
    messages = session_history + [{"role": "user", "content": user_request}]
    while True:
        blocks, stop_seen = {}, False
        with client.messages.stream(model=model, max_tokens=4096, tools=tools,
                                    messages=messages, thinking={"type": "adaptive"}) as stream:
            for event in stream:
                if event.type == "content_block_start":   blocks[event.index] = init_block(event)
                elif event.type == "content_block_delta": apply_delta(blocks[event.index], event.delta)
                elif event.type == "message_stop":        stop_seen = True
        assistant_content = [b for b in assemble(blocks) if b["type"] != "thinking"]
        messages.append({"role": "assistant", "content": assistant_content})
        response = finalize(blocks)
        if response.stop_reason == "end_turn":
            return response
        for block in response.content:
            if block.type == "tool_use":
                result = execute_tool(block.name, block.input)
                messages.append({"role": "user", "content": [{"type": "tool_result",
                                 "tool_use_id": block.id, "content": result}]})

def build_session_history(prior_sessions):
    full_history = []
    for session in prior_sessions:
        full_history.extend(session["messages"])      # every prior transcript, in-context
    return full_history
```

<details><summary><b>Show the answers</b></summary>

| Layer | Bug | Runtime effect | Fix |
|---|---|---|---|
| **Schema** | Description is just `"Gets data."` (no when / when-not / returns) | Claude can't route reliably; wrong or missed tool calls | 3–4 sentence description with an exclusion condition |
| **Streaming** | `stop_seen` is set but never checked before `messages.append` | A dropped stream commits a half-built `tool_use` to history → next request fails | Commit only if `stop_seen`; otherwise discard and retry |
| **Context** | Thinking blocks are filtered out of the assistant turn (and each tool result goes in its own user message) | Signature check fails — request rejected; multiple `tool_use` results must share **one** user turn | Append `assistant_content` unchanged; collect all results into one user message |
| **Memory** | All prior session transcripts concatenated in-context | Tokens + latency grow every session until the window fills | External storage + summarized/relevant subset at session start |

</details>

## 🏁 Cheat Sheet

| Chapter | One-line takeaway |
|---|---|
| Prompting | Diagnose the failure type, add the one missing technique; use structured outputs when the parser must never break |
| Thinking | On for hard multi-step work, off for lookups; return thinking blocks unchanged |
| Tools | Descriptions drive selection — say when *not* to use; every `tool_use` needs a matching `tool_result` in the next user turn |
| Streaming | Stream end ≠ message complete; commit only after `message_stop` |
| Context | Measure real tool outputs; prune, compact (with a good summarizer), cache stable prefixes, hand off to subagents |
| Agents | Workflow vs agent first; gate irreversible actions with a human; explicit exit conditions |
| Memory | Choose scope at design time; Skills load on demand |
| Multimodal/Batch | Price images before building; batch ≠ looping; join by `custom_id` |

**Next:** *Claude Code, MCP & Integration* — take this working integration and make it configurable, shareable and safe.